<a href="https://colab.research.google.com/github/Minimire/CROP-RECOMMENDATION-app/blob/main/Copy_of_%E1%88%9E%E1%8C%81%E1%88%8D%E1%8D%AE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# SINGLE-CELL COLAB WORKFLOW
# Creates model/artifacts, calculates metrics and PSI, logs/registers with MLflow,
# downloads the model bundle, and launches Streamlit through a temporary Cloudflare URL.

import os
import sys
import time
import json
import shutil
import subprocess
import urllib.request
from pathlib import Path

# Install dependencies in this Colab runtime.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "xgboost", "shap", "mlflow", "joblib", "scikit-learn",
        "pandas", "numpy", "matplotlib", "seaborn", "streamlit"
    ],
    check=True
)

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, top_k_accuracy_score
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_validate, train_test_split
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

# -------------------------
# 1. Configuration and data
# -------------------------
RANDOM_STATE = 42
FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
TARGET = "label"
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

csv_path = Path("Crop_recommendation.csv")
xlsx_path = Path("Crop_recommendation.xlsx")
data_url = (
    "https://raw.githubusercontent.com/Gladiator07/Harvestify/"
    "master/Data-processed/crop_recommendation.csv"
)

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"Loaded local file: {csv_path}")
elif xlsx_path.exists():
    df = pd.read_excel(xlsx_path)
    print(f"Loaded local file: {xlsx_path}")
else:
    df = pd.read_csv(data_url)
    print("Loaded dataset from the public CSV URL.")

required_columns = FEATURES + [TARGET]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

df = df[required_columns].dropna().copy()

if df.empty:
    raise ValueError("No rows remain after removing missing values.")

X = df[FEATURES].copy()
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[TARGET])
class_names = list(label_encoder.classes_)

print(f"\nDataset shape: {df.shape}")
print(f"Missing values after cleaning: {int(df.isna().sum().sum())}")
print(f"Duplicate rows: {int(df.duplicated().sum())}")
print(f"Crop classes: {len(class_names)}")
display(df[TARGET].value_counts().sort_index().rename("count").to_frame())

# -------------------------
# 2. Split data
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

# All models use a Pipeline, so scaling is fit only on each training fold.
def make_pipeline(classifier):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", classifier),
    ])

# -------------------------
# 3. Compare README models
# -------------------------
models = {
    "Random Forest": make_pipeline(
        RandomForestClassifier(
            n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1
        )
    ),
    "XGBoost": make_pipeline(
        XGBClassifier(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=5,
            objective="multi:softprob",
            num_class=len(class_names),
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "K-Nearest Neighbors (k=5)": make_pipeline(
        KNeighborsClassifier(n_neighbors=5)
    ),
    "Logistic Regression": make_pipeline(
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    ),
    "Dummy Classifier": make_pipeline(
        DummyClassifier(strategy="most_frequent")
    ),
}

comparison_rows = []
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    predictions = estimator.predict(X_test)
    comparison_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision (macro)": precision_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "Recall (macro)": recall_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "F1 (macro)": f1_score(
            y_test, predictions, average="macro", zero_division=0
        ),
    })

comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("Accuracy", ascending=False)
    .reset_index(drop=True)
)
print("\nModel comparison on the held-out test split:")
display(comparison.style.format({
    "Accuracy": "{:.2%}",
    "Precision (macro)": "{:.2%}",
    "Recall (macro)": "{:.2%}",
    "F1 (macro)": "{:.2%}",
}))
comparison.to_csv(ARTIFACT_DIR / "model_comparison.csv", index=False)

# -------------------------
# 4. Tune Random Forest
# -------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_pipeline = make_pipeline(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
)
param_grid = {
    "classifier__n_estimators": [100, 200, 500],
    "classifier__max_depth": [None, 10, 20],
    "classifier__max_features": ["sqrt", "log2"],
}
grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring={
        "accuracy": "accuracy",
        "f1_macro": "f1_macro",
        "recall_macro": "recall_macro",
    },
    refit="accuracy",
    cv=cv,
    n_jobs=-1,
    return_train_score=False,
    verbose=0,
)
grid.fit(X_train, y_train)
final_model = grid.best_estimator_

print("\nBest Random Forest parameters:")
print(grid.best_params_)
print(f"Best 5-fold CV accuracy: {grid.best_score_:.4f}")

# Get fold metrics recorded during grid search for the selected configuration.
best_index = grid.best_index_
fold_rows = []
for fold_index in range(cv.get_n_splits()):
    fold_rows.append({
        "Fold": fold_index + 1,
        "Accuracy": grid.cv_results_[f"split{fold_index}_test_accuracy"][best_index],
        "Macro F1": grid.cv_results_[f"split{fold_index}_test_f1_macro"][best_index],
        "Macro Recall": grid.cv_results_[f"split{fold_index}_test_recall_macro"][best_index],
    })
fold_table = pd.DataFrame(fold_rows)
fold_table.to_csv(ARTIFACT_DIR / "cross_validation_folds.csv", index=False)
print("\nSelected configuration's CV fold scores:")
display(fold_table.round(4))

# -------------------------
# 5. Final held-out test evaluation
# -------------------------
test_predictions = final_model.predict(X_test)
test_probabilities = final_model.predict_proba(X_test)
class_indices = np.arange(len(class_names))

test_metrics = {
    "accuracy": float(accuracy_score(y_test, test_predictions)),
    "precision_macro": float(
        precision_score(y_test, test_predictions, average="macro", zero_division=0)
    ),
    "recall_macro": float(
        recall_score(y_test, test_predictions, average="macro", zero_division=0)
    ),
    "f1_macro": float(
        f1_score(y_test, test_predictions, average="macro", zero_division=0)
    ),
    "top_3_accuracy": float(top_k_accuracy_score(
        y_test,
        test_probabilities,
        k=min(3, len(class_names)),
        labels=class_indices,
    )),
}

print("\nHeld-out test metrics:")
display(pd.DataFrame([test_metrics]).T.rename(columns={0: "Value"}).round(4))

report = classification_report(
    y_test,
    test_predictions,
    labels=class_indices,
    target_names=class_names,
    zero_division=0,
)
print("\nClassification report:")
print(report)
(ARTIFACT_DIR / "classification_report.txt").write_text(report, encoding="utf-8")

# Inference-only latency; this is not full web-app or network latency.
latency_ms = []
sample_input = X_test.iloc[[0]]
final_model.predict(sample_input)  # warm-up
for _ in range(100):
    start = time.perf_counter()
    final_model.predict(sample_input)
    latency_ms.append((time.perf_counter() - start) * 1000)

latency_p95_ms = float(np.percentile(latency_ms, 95))
print(f"Model-only inference latency p95: {latency_p95_ms:.2f} ms")

# -------------------------
# 6. Confusion matrix artifact
# -------------------------
cm = confusion_matrix(y_test, test_predictions, labels=class_indices)
plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=True,
)
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted crop")
plt.ylabel("Actual crop")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "confusion_matrix.png", dpi=200)
plt.show()

# -------------------------
# 7. SHAP summary artifact
# -------------------------
try:
    shap_sample = X_test.sample(min(200, len(X_test)), random_state=RANDOM_STATE)
    shap_scaled = final_model.named_steps["scaler"].transform(shap_sample)
    rf_model = final_model.named_steps["classifier"]
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(shap_scaled)

    plt.figure()
    shap.summary_plot(
        shap_values,
        shap_scaled,
        feature_names=FEATURES,
        show=False,
        max_display=len(FEATURES),
    )
    plt.tight_layout()
    plt.savefig(
        ARTIFACT_DIR / "shap_summary.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()
    print("Saved SHAP summary plot.")
except Exception as exc:
    print("SHAP plot was skipped due to a compatibility/runtime issue:", repr(exc))

# -------------------------
# 8. PSI monitoring function
# -------------------------
def calculate_psi(reference, current, bins=10):
    """Calculate PSI per feature using reference quantile bins."""
    reference = pd.DataFrame(reference).copy()
    current = pd.DataFrame(current).copy()
    rows = []

    for column in reference.columns:
        ref_values = pd.to_numeric(reference[column], errors="coerce").dropna().to_numpy()
        cur_values = pd.to_numeric(current[column], errors="coerce").dropna().to_numpy()

        if len(ref_values) == 0 or len(cur_values) == 0:
            rows.append({"Feature": column, "PSI": np.nan, "Interpretation": "No data"})
            continue

        quantiles = np.linspace(0, 1, bins + 1)[1:-1]
        cut_points = np.unique(np.quantile(ref_values, quantiles))
        edges = np.concatenate(([-np.inf], cut_points, [np.inf]))

        ref_counts, _ = np.histogram(ref_values, bins=edges)
        cur_counts, _ = np.histogram(cur_values, bins=edges)

        epsilon = 1e-6
        ref_pct = np.clip(ref_counts / max(ref_counts.sum(), 1), epsilon, None)
        cur_pct = np.clip(cur_counts / max(cur_counts.sum(), 1), epsilon, None)
        psi = float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

        interpretation = (
            "Little change" if psi < 0.10
            else "Moderate change" if psi < 0.25
            else "Large change"
        )
        rows.append({
            "Feature": column,
            "PSI": psi,
            "Interpretation": interpretation,
        })

    return pd.DataFrame(rows)

# For real monitoring, put a recent, unlabeled production-input batch in
# production_inputs.csv with the same seven feature columns. If absent, the
# held-out test set is used only as a demonstration—not production monitoring.
production_path = Path("production_inputs.csv")
if production_path.exists():
    current_features = pd.read_csv(production_path)[FEATURES]
    psi_source = "production_inputs.csv"
else:
    current_features = X_test[FEATURES]
    psi_source = "held-out test set (demonstration only)"

psi_results = calculate_psi(X_train[FEATURES], current_features)
psi_results.to_csv(ARTIFACT_DIR / "psi_report.csv", index=False)
print(f"\nPSI report source: {psi_source}")
display(psi_results.round(4))

# -------------------------
# 9. Save model bundle and model card
# -------------------------
model_bundle = {
    "model": final_model,
    "features": FEATURES,
    "label_encoder": label_encoder,
    "class_names": class_names,
    "random_state": RANDOM_STATE,
    "best_params": grid.best_params_,
    "metrics": test_metrics,
}
model_path = ARTIFACT_DIR / "crop_recommendation_model.pkl"
joblib.dump(model_bundle, model_path)

model_card = f"""# Crop Recommendation Model Card

## Model
- Model: Random Forest classifier, wrapped in a StandardScaler + classifier pipeline
- Task: 22-class crop recommendation
- Input features: {", ".join(FEATURES)}
- Training rows: {len(X_train)}
- Held-out test rows: {len(X_test)}
- Dataset rows after removing missing values: {len(df)}
- Dataset origin: India-specific crop recommendation dataset

## Test-set results
- Accuracy: {test_metrics["accuracy"]:.4f}
- Macro precision: {test_metrics["precision_macro"]:.4f}
- Macro recall: {test_metrics["recall_macro"]:.4f}
- Macro F1: {test_metrics["f1_macro"]:.4f}
- Top-3 accuracy: {test_metrics["top_3_accuracy"]:.4f}
- Model-only inference p95: {latency_p95_ms:.2f} ms

## Tuning
- Method: GridSearchCV with stratified 5-fold cross-validation
- Best parameters: {json.dumps(grid.best_params_, default=str)}
- Best mean CV accuracy: {grid.best_score_:.4f}

## Intended use and limitations
This is an educational pilot for crop recommendation. The dataset is India-specific
and does not establish suitability for Ethiopian farms. Local agronomic data,
expert review, field validation, and calibration are required before agricultural
or production use. Model probabilities are not calibrated confidence intervals.

## Monitoring
PSI results are saved in `psi_report.csv`. The test-set PSI in this run is only a
demonstration. For operational monitoring, compare a real production-input batch
against a representative training/reference baseline.
"""
(ARTIFACT_DIR / "model_card.md").write_text(model_card, encoding="utf-8")
print("\nSaved model bundle:", model_path)
print("Saved model card:", ARTIFACT_DIR / "model_card.md")

# -------------------------
# 10. MLflow tracking and model registry
# -------------------------
# Local Colab SQLite tracking store. It is local to this runtime unless downloaded.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("crop-recommendation-module-6")
registered_model_name = "CropRecommendationRandomForest"

with mlflow.start_run(run_name="random-forest-gridsearch") as run:
    mlflow.log_param("model", "RandomForestClassifier")
    mlflow.log_param("dataset_rows", len(df))
    mlflow.log_param("training_rows", len(X_train))
    mlflow.log_param("test_rows", len(X_test))
    mlflow.log_param("cv_folds", 5)

    for key, value in grid.best_params_.items():
        mlflow.log_param(key, str(value))

    for key, value in test_metrics.items():
        mlflow.log_metric(key, value)

    mlflow.log_metric("best_cv_accuracy", float(grid.best_score_))
    mlflow.log_metric("model_inference_latency_p95_ms", latency_p95_ms)

    mlflow.log_artifact(str(ARTIFACT_DIR / "model_card.md"))
    mlflow.log_artifact(str(ARTIFACT_DIR / "model_comparison.csv"))
    mlflow.log_artifact(str(ARTIFACT_DIR / "classification_report.txt"))
    mlflow.log_artifact(str(ARTIFACT_DIR / "cross_validation_folds.csv"))
    mlflow.log_artifact(str(ARTIFACT_DIR / "psi_report.csv"))

    for artifact_file in [
        "confusion_matrix.png",
        "shap_summary.png",
        "grid_search_results.csv",
        "grid_search_results.txt",
    ]:
        path = ARTIFACT_DIR / artifact_file
        if path.exists():
            mlflow.log_artifact(str(path))

    # Registers a new model version in the local MLflow registry.
    mlflow.sklearn.log_model(
        sk_model=final_model,
        artifact_path="model",
        registered_model_name=registered_model_name,
    )
    mlflow_run_id = run.info.run_id

print("\nMLflow run ID:", mlflow_run_id)
print("MLflow tracking database: mlflow.db")
print("Registered model name:", registered_model_name)

try:
    client = mlflow.tracking.MlflowClient()
    versions = client.search_model_versions(f"name='{registered_model_name}'")
    if versions:
        latest = max(versions, key=lambda version: int(version.version))
        print(f"Registered model version: {latest.version}")
        print(f"Model registry status: {latest.status}")
except Exception as exc:
    print("Model was logged, but registry version lookup failed:", repr(exc))

# -------------------------
# 11. Create Streamlit app
# -------------------------
app_source = r'''
import joblib
import numpy as np
import pandas as pd
import streamlit as st
from pathlib import Path

BUNDLE_PATH = Path("artifacts/crop_recommendation_model.pkl")
bundle = joblib.load(BUNDLE_PATH)
model = bundle["model"]
features = bundle["features"]
label_encoder = bundle["label_encoder"]

st.set_page_config(page_title="Crop Recommendation", page_icon="🌾")
st.title("🌾 Intelligent Crop Recommendation System")
st.caption("Educational pilot. Validate recommendations with a local agronomist.")

with st.form("crop_inputs"):
    col1, col2 = st.columns(2)
    values = {}
    for index, feature in enumerate(features):
        target_col = col1 if index % 2 == 0 else col2
        defaults = {
            "N": (90.0, 0.0, 200.0),
            "P": (42.0, 0.0, 200.0),
            "K": (43.0, 0.0, 250.0),
            "temperature": (20.8, -10.0, 60.0),
            "humidity": (82.0, 0.0, 100.0),
            "ph": (6.5, 0.0, 14.0),
            "rainfall": (202.9, 0.0, 500.0),
        }
        default, minimum, maximum = defaults[feature]
        with target_col:
            values[feature] = st.number_input(
                feature,
                min_value=float(minimum),
                max_value=float(maximum),
                value=float(default),
            )
    submitted = st.form_submit_button("Recommend crop")

if submitted:
    input_frame = pd.DataFrame([[values[name] for name in features]], columns=features)
    prediction = model.predict(input_frame)[0]
    crop = label_encoder.inverse_transform([int(prediction)])[0]
    probabilities = model.predict_proba(input_frame)[0]
    probability = float(np.max(probabilities))

    st.success(f"Recommended crop: **{crop}**")
    st.write(f"Model probability for this prediction: **{probability:.1%}**")
    st.warning(
        "This probability is not a calibrated confidence interval. "
        "The training dataset is India-specific; local Ethiopian validation is needed."
    )
'''

Path("app.py").write_text(app_source, encoding="utf-8")
print("\nCreated app.py")

# -------------------------
# 12. Start Streamlit and a temporary Cloudflare Tunnel
# -------------------------
# Cloudflare quick tunnels provide temporary URLs and are not permanent hosting.
machine = os.uname().machine
if machine not in ("x86_64", "amd64"):
    raise RuntimeError(
        f"Automatic Cloudflare setup expects x86_64 Colab; detected {machine}. "
        "You can still download the model and run app.py separately."
    )

cloudflared_path = Path("/content/cloudflared")
cloudflared_url = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64"
)

if not cloudflared_path.exists():
    urllib.request.urlretrieve(cloudflared_url, cloudflared_path)
    cloudflared_path.chmod(0o755)

streamlit_log = open("streamlit.log", "w")
streamlit_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)

time.sleep(8)
if streamlit_process.poll() is not None:
    streamlit_log.flush()
    raise RuntimeError("Streamlit did not start. Check streamlit.log for details.")

tunnel_log = open("cloudflared.log", "w")
tunnel_process = subprocess.Popen(
    [
        str(cloudflared_path),
        "tunnel",
        "--url", "http://127.0.0.1:8501",
        "--no-autoupdate",
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

live_url = None
for _ in range(60):
    time.sleep(2)
    tunnel_log.flush()
    log_text = Path("cloudflared.log").read_text(
        encoding="utf-8", errors="ignore"
    )
    import re
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        live_url = match.group(0)
        break
    if tunnel_process.poll() is not None:
        break

if live_url:
    print("\nLIVE APPLICATION URL (temporary):")
    print(live_url)
    display(Markdown(f"[Open the crop recommendation app]({live_url})"))
    print("Keep this Colab runtime running while using the app.")
else:
    print("Could not detect the temporary Cloudflare URL.")
    print("Check cloudflared.log and streamlit.log.")
    print("The model, metrics, and artifacts have still been created.")

# -------------------------
# 13. Download model and artifacts
# -------------------------
try:
    from google.colab import files

    print("\nStarting download of the model bundle...")
    files.download(str(model_path))
except Exception as exc:
    print("\nAutomatic Colab download was unavailable:", repr(exc))
    print("Download this file from the Colab Files panel:", model_path)

# Also package all report artifacts into one downloadable ZIP.
zip_path = shutil.make_archive("crop_recommendation_artifacts", "zip", ARTIFACT_DIR)
try:
    from google.colab import files
    print("Starting download of the artifact ZIP...")
    files.download(zip_path)
except Exception:
    print("Artifact ZIP saved at:", zip_path)

print("\nCompleted. Main outputs:")
print("- Model bundle: artifacts/crop_recommendation_model.pkl")
print("- Model card: artifacts/model_card.md")
print("- PSI report: artifacts/psi_report.csv")
print("- Grid-search results: artifacts/grid_search_results.txt")
print("- Confusion matrix: artifacts/confusion_matrix.png")
print("- SHAP summary, if generated: artifacts/shap_summary.png")
print("- Streamlit app source: app.py")
print("- MLflow database: mlflow.db")